<a href="https://www.kaggle.com/code/flixrojas/air-quality-data-filtering?scriptVersionId=312411690" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client()
dataset_ref = client.dataset("epa_historical_air_quality", project="bigquery-public-data")
dataset = client.get_dataset(dataset_ref)

tables = list(client.list_tables(dataset))

print("Tables in the EPA dataset:")
for table in tables:
    print(table.table_id)

In [ ]:
table_ref = dataset_ref.table("air_quality_annual_summary")
table = client.get_table(table_ref)

for schema_field in table.schema:
    print(f"{schema_field.name} - {schema_field.field_type}")

In [ ]:
# Extracting the arithmetic mean of particulates below 2.5
# these particulates stem from vehicle and building emissions
# that can be generated by burning, chemical reactions, dust storms
# they can also be secondarily generated in the atmosphere through chemical reactions

query = """
SELECT 
    date_local, 
    arithmetic_mean as pm25
FROM 
    `bigquery-public-data.epa_historical_air_quality.pm25_frm_daily_summary`
WHERE 
    state_name = 'California' 
    AND county_name = 'Los Angeles'
    AND date_local >= '2018-01-01' AND date_local <= '2025-12-31'
ORDER BY 
    date_local ASC
"""

query_job = client.query(query)
 
df = query_job.to_dataframe()

display(df.head())

I'm changing the format to pandas datetime just to make sure it is handled properly

I will also be printing the dataframe info to make sure I do not have to deal with empty or null values since it is a time series

In [ ]:
# reformatting the data and averaging county data measurements
df['date_local'] = pd.to_datetime(df['date_local'])

# date is now the index so we can query using a specified datetime
df.set_index('date_local', inplace=True)

print(df.info())

## Preprocessing
Since the data is a time series, we need to ensure that it is:
- Contiguous
- Sorted
- Free of missing values (done)

I will proceed to visualize the dataset to look for 
- Seasonality (repeating patterns)
- Trend (arrangement of datpoints following a pattern)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.plot(df.index, df['pm25'], label='Daily PM2.5', linewidth=1)
plt.title('Daily Average PM2.5 in LA (2018-2025)')
plt.xlabel('Date')
plt.ylabel('PM2.5 Concentration (µg/m³)')
plt.legend()
plt.show()

That is how the overall data looks but there are two problems: 
- sample is too large, so I will reduce it to a random year to see if there might be seasonal changes (like summer wildfires)
- Datapoints from 2022 - 2023, 2023 - 2024 and 2024-2025 look suspiciously wrong 

First I will plot two years that don't look suspicious to see if there is seasonality or trend within them. I'll define a function to plot a range for this PDA.

In [ ]:
def plot_date_range(df, start_date, end_date, column='pm25'):
    date_range_subset = df.loc[pd.to_datetime(start_date):pd.to_datetime(end_date)]
    
    plt.figure(figsize=(15, 6))
    plt.plot(date_range_subset.index, date_range_subset[column], linewidth=1.5)
    
    plt.title(f'Daily Average PM2.5 in Los Angeles ({start_date} to {end_date})')
    plt.xlabel('Date')
    plt.ylabel('PM2.5 Concentration (µg/m³)')
    plt.tight_layout()
    plt.show()

plot_date_range(df, '2018-01-01','2018-12-31')

In [ ]:
plot_date_range(df, '2019-01-01','2019-12-31')

In [ ]:
plot_date_range(df, '2020-01-01','2020-12-31')

There seems to be some seasonality on the last months and the beginning months (probably due to increased electric consumption in winter) and an insane peak in July (fireworks due to July 4th)

I will now make boxplots for the seasons to make the visualization clearer

In [ ]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

def plot_seasonal_buckets(df, start_date, end_date, column='pm25'):
    subset = df.loc[start_date:end_date].copy()
    
    subset['season'] = subset.index.month.map(get_season)
    plt.figure(figsize=(10, 6))
    season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
    
    sns.boxplot(data=subset, x='season', y=column, order=season_order)
    
    plt.title(f'Seasonal PM2.5 Buckets in Los Angeles ({start_date} to {end_date})')
    plt.xlabel('Season')
    plt.ylabel('PM2.5 Concentration (µg/m³)')
    plt.tight_layout()
    plt.show()

plot_seasonal_buckets(df, '2018-01-01', '2018-12-31')

In [ ]:
# the following code was generated by Gemini 3.1 pro
# why? i needed a complex graph and I am not used to the seaborn package
# it is also just for EDA

# 1. Create a copy and extract the necessary categorical columns
df_subplots = df.copy()
df_subplots['year'] = df_subplots.index.year
df_subplots['season'] = df_subplots.index.month.map(get_season)

# 2. Define the logical order for the x-axis
season_order = ['Winter', 'Spring', 'Summer', 'Autumn']

# 3. Use catplot to generate the subplots
# 'col="year"' tells Seaborn to make a new subplot for each unique year
# 'col_wrap=3' limits the grid to 3 subplots per row before wrapping to the next line
g = sns.catplot(
    data=df_subplots, 
    x='season', 
    y='pm25', 
    col='year', 
    col_wrap=3, 
    kind='box', 
    order=season_order,
    height=4, 
    aspect=1.2,
    palette='Set2'
)

# 4. Format the titles and labels
g.set_axis_labels("Season", "PM2.5 Concentration (µg/m³)")
g.set_titles("Year: {col_name}")

# Add a main title to the entire figure (y=1.05 pushes it slightly above the subplots)
g.fig.suptitle("Seasonal Distribution of PM2.5 in Los Angeles (2018-2022)", y=1.05)

plt.show()

Years 2022 and onwards look incomplete, so I will only use the data from 2018-2021.

Also there is a clear indication that Winter has the largest variation and ranges for PM 2.5, so this set does possess seasonality (as well as in Autumn)!

## Data split
Since the data from 2022 seems to be incomplete, I will split the data using the classic 80 - 20 split from 2018 to 2021

In [ ]:
filtered_data = df.loc['2018-01-01':'2021-12-31']

training_fraction = 0.8

# index where the split happens
# int casting because it's supposed to be an idx
split_index = int(training_fraction * len(filtered_data))

train_data = filtered_data.iloc[:split_index]
test_data = filtered_data.iloc[split_index:]

Finally we rescale the data to go from 0.0 to 1.0 for training

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))

train_scaled = scaler.fit_transform(train_data[['pm25']])
test_scaled = scaler.transform(test_data[['pm25']])

## Second approximation
My goal is to determine PM2.5 presence in the air using the overall data from 2018-2021, due to some data gaps in 2023
I will do the same query but merging related data to wind and temperature as potentially predictive

I'll do a test query to see how many datapoints we are dealing with

In [ ]:
test_query = """
SELECT
    pm.state_code,
    pm.county_code,
    pm.site_num,
    pm.date_local,
    pm.arithmetic_mean AS pm25_level,
    t.arithmetic_mean AS temperature,
    w.arithmetic_mean AS wind_speed
FROM
    `bigquery-public-data.epa_historical_air_quality.pm25_frm_daily_summary` AS pm
INNER JOIN
    `bigquery-public-data.epa_historical_air_quality.temperature_daily_summary` AS t
    ON pm.state_code = t.state_code
    AND pm.county_code = t.county_code
    AND pm.site_num = t.site_num
    AND pm.date_local = t.date_local
INNER JOIN
    `bigquery-public-data.epa_historical_air_quality.wind_daily_summary` AS w
    ON pm.state_code = w.state_code
    AND pm.county_code = w.county_code
    AND pm.site_num = w.site_num
    AND pm.date_local = w.date_local
WHERE
    -- only 2020
    pm.date_local >= '2020-01-01' 
    AND pm.date_local <= '2020-12-31'
    AND pm.state_code = '06' -- California
ORDER BY
    pm.state_code,
    pm.county_code,
    pm.site_num,
    pm.date_local;
"""

query_job = client.query(test_query)
 
df_large_test = query_job.to_dataframe()

display(df_large_test.head())

In [ ]:
df_large_test.info()

## Nation-wide query
78434 datapoints is nice but I want a nationwide datasweep to determine how wind and temperature carry PM2.5

In [ ]:
nation_wide_query = """
SELECT
    pm.state_code,
    pm.county_code,
    pm.site_num,
    pm.date_local,
    pm.time_local,
    pm.sample_measurement AS pm25_level,
    t.sample_measurement AS temperature,
    w.sample_measurement AS wind_speed
FROM
    `bigquery-public-data.epa_historical_air_quality.pm25_frm_hourly_summary` AS pm
INNER JOIN
    `bigquery-public-data.epa_historical_air_quality.temperature_hourly_summary` AS t
    ON pm.state_code = t.state_code
    AND pm.county_code = t.county_code
    AND pm.site_num = t.site_num
    AND pm.date_local = t.date_local
    AND pm.time_local = t.time_local
INNER JOIN
    `bigquery-public-data.epa_historical_air_quality.wind_hourly_summary` AS w
    ON pm.state_code = w.state_code
    AND pm.county_code = w.county_code
    AND pm.site_num = w.site_num
    AND pm.date_local = w.date_local
    AND pm.time_local = w.time_local
WHERE
    -- only 2020
    pm.date_local >= '2020-01-01' 
    AND pm.date_local <= '2020-12-31'
ORDER BY
    pm.state_code,
    pm.county_code,
    pm.site_num,
    pm.time_local,
    pm.date_local;
"""
query_job = client.query(nation_wide_query)
 
usa_2020_aq_df = query_job.to_dataframe()

display(usa_2020_aq_df.head())

In [ ]:
usa_2020_aq_df.info()

In [ ]:
# convert to pandas format
usa_2020_aq_df['date_local'] = pd.to_datetime(usa_2020_aq_df['date_local'].astype(str))
usa_2020_aq_df['timestamp'] = pd.to_datetime(usa_2020_aq_df['date_local'].astype(str) + ' ' + usa_2020_aq_df['time_local'])

usa_2020_aq_df.info()

In [ ]:
# save to compressed parquet file for backup
usa_2020_aq_df.to_parquet('usa_2020_aq_df.parquet', engine='pyarrow')

In [ ]:
# make the same process for years 2018-2021

def save_aq_df_2018_2021():
    nation_wide_query = """
SELECT
    pm.state_code,
    pm.county_code,
    pm.site_num,
    pm.date_local,
    pm.time_local,
    pm.sample_measurement AS pm25_level,
    t.sample_measurement AS temperature,
    w.sample_measurement AS wind_speed
FROM
    `bigquery-public-data.epa_historical_air_quality.pm25_frm_hourly_summary` AS pm
INNER JOIN
    `bigquery-public-data.epa_historical_air_quality.temperature_hourly_summary` AS t
    ON pm.state_code = t.state_code
    AND pm.county_code = t.county_code
    AND pm.site_num = t.site_num
    AND pm.date_local = t.date_local
    AND pm.time_local = t.time_local
INNER JOIN
    `bigquery-public-data.epa_historical_air_quality.wind_hourly_summary` AS w
    ON pm.state_code = w.state_code
    AND pm.county_code = w.county_code
    AND pm.site_num = w.site_num
    AND pm.date_local = w.date_local
    AND pm.time_local = w.time_local
WHERE
    pm.date_local >= '2018-01-01' 
    AND pm.date_local <= '2021-12-31'
ORDER BY
    pm.state_code,
    pm.county_code,
    pm.site_num,
    pm.time_local,
    pm.date_local;
"""
    query_job = client.query(nation_wide_query)
    usa_2020_aq_df = query_job.to_dataframe()
    
    # convert to pandas format
    usa_2020_aq_df['date_local'] = pd.to_datetime(usa_2020_aq_df['date_local'].astype(str))
    usa_2020_aq_df['timestamp'] = pd.to_datetime(usa_2020_aq_df['date_local'].astype(str) + ' ' + usa_2020_aq_df['time_local'])
    usa_2020_aq_df.to_parquet('usa_2018_2021_aq_df.parquet', engine='pyarrow')

save_aq_df_2018_2021()

In [ ]:
backup_df = pd.read_parquet("/kaggle/working/usa_2018_2021_aq_df.parquet")

In [ ]:
# make the same process for years 2018-2021

def save_aq_df_2018_2021_geospatial():
    nation_wide_geoquery = """
SELECT
    pm.state_code,
    pm.county_code,
    pm.site_num,
    pm.latitude,
    pm.longitude,
    pm.date_local,
    pm.time_local,
    pm.sample_measurement AS pm25_level,
    t.sample_measurement AS temperature,
    w.sample_measurement AS wind_speed
FROM
    `bigquery-public-data.epa_historical_air_quality.pm25_frm_hourly_summary` AS pm
INNER JOIN
    `bigquery-public-data.epa_historical_air_quality.temperature_hourly_summary` AS t
    ON pm.state_code = t.state_code
    AND pm.county_code = t.county_code
    AND pm.site_num = t.site_num
    AND pm.date_local = t.date_local
    AND pm.time_local = t.time_local
INNER JOIN
    `bigquery-public-data.epa_historical_air_quality.wind_hourly_summary` AS w
    ON pm.state_code = w.state_code
    AND pm.county_code = w.county_code
    AND pm.site_num = w.site_num
    AND pm.date_local = w.date_local
    AND pm.time_local = w.time_local
WHERE
    pm.date_local >= '2018-01-01' 
    AND pm.date_local <= '2021-12-31'
ORDER BY
    pm.state_code,
    pm.county_code,
    pm.latitude,
    pm.longitude,
    pm.site_num,
    pm.time_local,
    pm.date_local;
"""
    query_job = client.query(nation_wide_geoquery)
    usa_2020_aq_df_geo = query_job.to_dataframe()
    
    # convert to pandas format
    usa_2020_aq_df_geo['date_local'] = pd.to_datetime(usa_2020_aq_df_geo['date_local'].astype(str))
    usa_2020_aq_df_geo['timestamp'] = pd.to_datetime(usa_2020_aq_df_geo['date_local'].astype(str) + ' ' + usa_2020_aq_df_geo['time_local'])
    usa_2020_aq_df_geo.to_parquet('usa_2018_2021_aq_df_geoquery.parquet', engine='pyarrow')

save_aq_df_2018_2021_geospatial()

## Calculate the new split on the time series
I'm using a random split to predict spatial values based on adjacent values

In [2]:
import pandas as pd
backup_df = pd.read_parquet("/kaggle/input/datasets/flixrojas/usa-2018-2021-aq-df-geoquery-parquet")
backup_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15948841 entries, 0 to 15948840
Data columns (total 11 columns):
 #   Column       Dtype         
---  ------       -----         
 0   state_code   object        
 1   county_code  object        
 2   site_num     object        
 3   latitude     float64       
 4   longitude    float64       
 5   date_local   datetime64[ns]
 6   time_local   object        
 7   pm25_level   float64       
 8   temperature  float64       
 9   wind_speed   float64       
 10  timestamp    datetime64[ns]
dtypes: datetime64[ns](2), float64(5), object(4)
memory usage: 1.3+ GB


## We rescale the data for fitting

On my original approach I was just trying to calculate PM25 without taking time into account. This was a mistake and upon review of the general literature, the preprocessing for spatio-temporal features was lacking.

## Cyclical encoding

In [3]:
import numpy as np
import pandas as pd

day_of_year = backup_df['timestamp'].dt.dayofyear
hour_of_day = backup_df['timestamp'].dt.hour

DAYS_IN_YEAR = 365.2425 # leap year approximate
HOURS_IN_DAY = 24.0

backup_df['day_sin'] = np.sin(2 * np.pi * day_of_year / DAYS_IN_YEAR)
backup_df['day_cos'] = np.cos(2 * np.pi * day_of_year / DAYS_IN_YEAR)

backup_df['hour_sin'] = np.sin(2 * np.pi * hour_of_day / HOURS_IN_DAY)
backup_df['hour_cos'] = np.cos(2 * np.pi * hour_of_day / HOURS_IN_DAY)

## Features and target

In [4]:
lstm_features = ['latitude', 'longitude', 'temperature', 'wind_speed', 'day_sin', 'day_cos', 'hour_sin', 'hour_cos']
target_col = 'pm25_level'

### LSTM sequences
I'll group the cyclical time data, grouping it in 7 days 

In [6]:
def create_lstm_sequences(df, features, target, sequence_length=7):
    X_seq, y_seq = [], []

    # group by lat,lon pairs
    grouped = df.groupby(['latitude', 'longitude'])
    
    for _, group in grouped:
        group = group.sort_values('timestamp')
        
        feature_data = group[features].values
        target_data = group[target].values
        
        # create sliding windows
        for i in range(len(group) - sequence_length):
            X_seq.append(feature_data[i : i + sequence_length])
            # target PM2.5 level for the next timestep
            y_seq.append(target_data[i + sequence_length])
            
    return np.array(X_seq), np.array(y_seq)

sequence_length = 7
X_lstm, y_lstm = create_lstm_sequences(backup_df, lstm_features, target_col, sequence_length)

print(f"LSTM Input Shape: {X_lstm.shape}") # Should be (samples, 7, 8)
print(f"LSTM Target Shape: {y_lstm.shape}") # Should be (samples,)

LSTM Input Shape: (15946818, 7, 8)
LSTM Target Shape: (15946818,)


In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_lstm_model(sequence_length, num_features):
    model = models.Sequential([
        # 3d input
        layers.Input(shape=(sequence_length, num_features)),
        
        layers.LSTM(64, return_sequences=False),
        layers.BatchNormalization(),
        
        # Standard Dense layers for final regression mapping
        layers.Dense(32, activation='relu'),
        layers.Dense(16, activation='relu'),
        
        # continuous PM2.5 prediction
        layers.Dense(1)
    ])
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model

lstm_model = build_lstm_model(
    sequence_length=X_lstm.shape[1], 
    num_features=X_lstm.shape[2]
)

lstm_model.summary()

2026-04-17 19:13:36.833645: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776453217.087116    1328 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776453217.158037    1328 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776453217.773452    1328 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776453217.773492    1328 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776453217.773495    1328 computation_placer.cc:177] computation placer alr

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        18,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,569 (84.25 KB)

 Trainable params: 21,441 (83.75 KB)

 Non-trainable params: 128 (512.00 B)

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import train_test_split

# 80-20 split
X_train_seq, X_val_seq, y_train_seq, y_val_seq = train_test_split(
    X_lstm, y_lstm, test_size=0.2, random_state=1, shuffle=True
)

# lstm model 
checkpoint = ModelCheckpoint(
    filepath='pm25_lstm_model_checkpoint.keras', 
    monitor='val_loss',
    save_best_only=True,  
    mode='min',           
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True
)

history = lstm_model.fit(
    X_train_seq, 
    y_train_seq,
    epochs=40,
    batch_size=1024, 
    validation_data=(X_val_seq, y_val_seq),
    verbose=1,
    callbacks=[checkpoint, early_stop]
)

Epoch 1/40
12458/12459 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 112.5911 - mae: 4.8973
Epoch 1: val_loss improved from inf to 122.82846, saving model to pm25_lstm_model_checkpoint.keras
12459/12459 ━━━━━━━━━━━━━━━━━━━━ 389s 31ms/step - loss: 112.5907 - mae: 4.8973 - val_loss: 122.8285 - val_mae: 5.8725
Epoch 2/40
 7031/12459 ━━━━━━━━━━━━━━━━━━━━ 2:25 27ms/step - loss: 109.8027 - mae: 4.6949

In [ ]:
import numpy as np
from datetime import datetime

def predict_pm25(model, scaler_x, scaler_y, latitude, longitude, target_date, temperature, wind_speed):
    day_of_year = target_date.timetuple().tm_yday
    hour_of_day = target_date.hour
    
    DAYS_IN_YEAR = 365.2425
    HOURS_IN_DAY = 24.0
    
    day_sin = np.sin(2 * np.pi * day_of_year / DAYS_IN_YEAR)
    day_cos = np.cos(2 * np.pi * day_of_year / DAYS_IN_YEAR)
    hour_sin = np.sin(2 * np.pi * hour_of_day / HOURS_IN_DAY)
    hour_cos = np.cos(2 * np.pi * hour_of_day / HOURS_IN_DAY)
    
    raw_input = np.array([[
        latitude, 
        longitude, 
        temperature, 
        wind_speed, 
        day_sin, 
        day_cos, 
        hour_sin, 
        hour_cos
    ]])

    scaled_input = scaler_x.transform(raw_input)
    scaled_prediction = model.predict(scaled_input, verbose=0)
    
    # Inverse transform the output to get the real PM2.5 concentration in µg/m³
    actual_prediction = scaler_y.inverse_transform(scaled_prediction)
    
    return actual_prediction[0][0]

In [ ]:
# Example: Predicting PM 2.5 in Los Angeles for July 4th, 2024 at 2:00 PM
# Assuming 25°C temperature and 5 mph wind speed

target_datetime = datetime(2024, 7, 4, 14, 0, 0)
la_lat = 34.0522
la_lon = -118.2437
forecast_temp = 25.0
forecast_wind = 5.0

predicted_level = predict_pm25(
    model=model, 
    scaler_x=scaler_x, 
    scaler_y=scaler_y, 
    latitude=la_lat, 
    longitude=la_lon, 
    target_date=target_datetime, 
    temperature=forecast_temp, 
    wind_speed=forecast_wind
)

print(f"Predicted PM2.5: {predicted_level:.2f} µg/m³")

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred_seq_scaled = lstm_model.predict(X_val_seq)

# sequence generator outputs (samples,), but scaler_y expects (samples, 1)
y_val_seq_2d = y_val_seq.reshape(-1, 1)

# get actual PM2.5 levels (µg/m³)
y_pred_seq_actual = scaler_y.inverse_transform(y_pred_seq_scaled)
y_true_seq_actual = scaler_y.inverse_transform(y_val_seq_2d)

mae_lstm = mean_absolute_error(y_true_seq_actual, y_pred_seq_actual)
mse_lstm = mean_squared_error(y_true_seq_actual, y_pred_seq_actual)
rmse_lstm = np.sqrt(mse_lstm)
r2_lstm = r2_score(y_true_seq_actual, y_pred_seq_actual)

# 5. Display the results
print(f"LSTM Real-World MAE:  {mae_lstm:.2f} µg/m³")
print(f"LSTM Real-World MSE:  {mse_lstm:.2f} (µg/m³)²")
print(f"LSTM Real-World RMSE: {rmse_lstm:.2f} µg/m³")
print(f"LSTM R-squared (R2):  {r2_lstm:.4f}")